In [ ]:
import os
import pydicom
import h5py
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn.functional as F

In [ ]:
import os
import pydicom
import h5py
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


def make_dir(path, exist_ok=True):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=exist_ok)
        os.chmod(path, 0o777)


def resize_volume_torch(volume_np, target_shape=(192, 192)):
    vol = torch.tensor(volume_np, dtype=torch.float32).unsqueeze(1)  # (Z, 1, H, W)
    vol_resized = F.interpolate(vol, size=target_shape, mode='bilinear', align_corners=False)
    return vol_resized.squeeze(1).numpy().astype(np.int16)  # (Z, H, W)


def get_bvalue(ds):
    """DWI b-value 추출. Siemens private tag(0019,100c)를 우선 쓰고,
    없으면 표준 DiffusionBValue 태그를 시도한다. 둘 다 없으면 -1(unknown)."""
    tag = ds.get((0x0019, 0x100c), None)
    if tag is not None:
        try:
            return int(str(tag.value).split('\\')[0])
        except (ValueError, TypeError):
            pass
    bval = getattr(ds, 'DiffusionBValue', None)
    if bval is not None:
        try:
            return int(bval)
        except (ValueError, TypeError):
            pass
    return -1


def load_dicom_volume(dicom_folder, target_shape=(192, 192), extra_info=None):
    all_files = sorted([f for f in os.listdir(dicom_folder) if f.endswith(".dcm")])
    slices, instance_numbers, file_paths, bvalues = [], [], [], []
    extra_data = {key: [] for key in extra_info} if extra_info else {}

    for f in all_files:
        path = os.path.join(dicom_folder, f)
        ds = pydicom.dcmread(path)
        slices.append(ds.pixel_array)
        instance_numbers.append(int(ds.InstanceNumber))
        file_paths.append(path)
        bvalues.append(get_bvalue(ds))

        if extra_info is not None:
            for key in extra_info:
                value = getattr(ds, key, "")
                if isinstance(value, pydicom.multival.MultiValue):
                    value = list(value)
                extra_data[key].append(value)

    sorted_indices = np.argsort(instance_numbers)
    slices = [slices[i] for i in sorted_indices]
    file_paths = [file_paths[i] for i in sorted_indices]
    bvalues = [bvalues[i] for i in sorted_indices]
    for key in extra_data:
        extra_data[key] = [extra_data[key][i] for i in sorted_indices]
    extra_data['b_value'] = bvalues

    volume = np.stack(slices)
    volume_resized = resize_volume_torch(volume, target_shape)
    return volume_resized, file_paths, extra_data


def resize_mask_volume_torch(volume_np, target_shape=(192, 192)):
    vol = torch.tensor(volume_np, dtype=torch.float32).unsqueeze(1)  # (Z, 1, H, W)
    vol_resized = F.interpolate(vol, size=target_shape, mode='nearest')
    return vol_resized.squeeze(1).numpy().astype(np.uint8)


def load_mask_volume(labelmask_folder, dcm_files, target_shape=(192, 192)):
    mask_volume = []
    for f in dcm_files:
        base = os.path.splitext(os.path.basename(f))[0]
        png_path = os.path.join(labelmask_folder, f"{base}.png")
        if os.path.exists(png_path):
            mask = Image.open(png_path).convert("L")
            mask_np = (np.array(mask) > 0).astype(np.uint8)
        else:
            mask_np = None
        mask_volume.append(mask_np)

    default_shape = next((m.shape for m in mask_volume if m is not None), target_shape)
    for i in range(len(mask_volume)):
        if mask_volume[i] is None:
            mask_volume[i] = np.zeros(default_shape, dtype=np.uint8)

    stacked = np.stack(mask_volume)
    return resize_mask_volume_torch(stacked, target_shape)


def resolve_case_dir(patient_dir, case):
    """환자마다 'pre'/'post' 또는 'preop'/'postop' 중 실제 존재하는 폴더명을 찾는다."""
    for name in (case, case + "op"):
        p = os.path.join(patient_dir, name)
        if os.path.isdir(p):
            return p
    return None


def get_pairs(patient_dir, extra_info=None):
    """pre/post를 각각 독립적으로 처리. 해당 case의 dicom 폴더 또는 labelmask 폴더가
    없으면 그 case만 건너뛴다 (라벨 없는 case를 '정상'으로 임의 추정하지 않음)."""
    results = {}
    for case in ['pre', 'post']:
        input_path = resolve_case_dir(patient_dir, case)
        label_path = os.path.join(patient_dir, "labelmask", case)

        if input_path is None:
            continue
        if not os.path.isdir(label_path):
            continue

        dcm_volume, dcm_files, extra_data = load_dicom_volume(input_path, extra_info=extra_info)
        mask_volume = load_mask_volume(label_path, dcm_files)
        results[case] = (dcm_files, dcm_volume, mask_volume, extra_data)

    return results


def get_h5(root_path, out_root=None, extra_info=None, limit_patients=None):
    out_root = out_root or os.path.join(root_path, 'h5')
    n_saved = 0
    n_skipped = 0
    n_patients = 0

    for type_id in sorted(os.listdir(root_path)):
        type_path = os.path.join(root_path, type_id)
        if not os.path.isdir(type_path) or type_id == os.path.basename(out_root):
            continue

        for patient_id in sorted(os.listdir(type_path)):
            patient_dir = os.path.join(type_path, patient_id)
            if not os.path.isdir(patient_dir):
                continue

            if limit_patients is not None and n_patients >= limit_patients:
                return n_saved, n_skipped

            n_patients += 1
            pairs = get_pairs(patient_dir, extra_info=extra_info)
            if not pairs:
                print(f'[skip] no usable pre/post + label pair: {patient_dir}')
                n_skipped += 1
                continue

            save_dir = os.path.join(out_root, type_id)
            make_dir(save_dir)

            for case, (dcm_files, dcm_volume, mask_volume, extra_data) in pairs.items():
                save_path = os.path.join(save_dir, f"{patient_id}_{case}.h5")
                with h5py.File(save_path, 'w') as f:
                    f.create_dataset('meta', data=dcm_files)
                    f.create_dataset('dcm', data=dcm_volume)
                    f.create_dataset('label', data=mask_volume)
                    f.create_dataset('b_value', data=np.array(extra_data['b_value']))
                    if extra_info:
                        for key in extra_info:
                            f.create_dataset(key, data=np.array(extra_data[key]))
                n_saved += 1
                print(f'[saved] {save_path}  slices={dcm_volume.shape[0]}  lesion_slices={(mask_volume.sum(axis=(1,2))>0).sum()}')

    return n_saved, n_skipped

In [ ]:
# repo 루트 기준 data/ (src/ 에서 실행 시 상대경로 "../data/...")
root_path = '../data/results_train_cleaned'
out_root = '../data/h5'
n_saved, n_skipped = get_h5(root_path, out_root=out_root, extra_info=['SliceThickness'])
print(f'saved={n_saved} skipped_patients={n_skipped}')

In [ ]:
f = h5py.File(list(__import__('glob').glob(out_root + '/**/*.h5', recursive=True))[0])


In [ ]:
f.keys()

In [ ]:
f['meta'][0].decode('utf-8')

In [ ]:
import pydicom

def resize_img_torch(img_np, target_shape=(192, 192)):
    img = torch.tensor(img_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
    resized = F.interpolate(img, size=target_shape, mode='bilinear', align_corners=False)
    return resized.squeeze().numpy().astype(np.int16)  # (H, W)

def dcm2img(d):
    dcm = pydicom.dcmread(d)
    return resize_img_torch(dcm.pixel_array)


In [ ]:

plt.subplot(1,3,1)
plt.imshow(f['dcm'][0], cmap='gray')
plt.subplot(1,3,2)
plt.imshow(dcm2img(f['meta'][0].decode('utf-8')), cmap='gray')
plt.subplot(1,3,3)
plt.imshow(f['dcm'][0] - dcm2img(f['meta'][0].decode('utf-8')), cmap ='bwr')
plt.colorbar()